# 🤖 Métodos de Machine Learning en Medicina
## Taller Pre-Congreso CNIB 2025 — Dataset Clínico Artificial

---

> **Contexto del notebook:** Aquí exploramos cuatro algoritmos clásicos de ML usando un dataset *sintético* de un ensayo clínico farmacológico. El objetivo es predecir si un paciente presentará efectos secundarios a un medicamento dado su edad, la dosis y su función renal. Trabajar con datos artificiales nos permite entender los algoritmos sin preocuparnos por el preprocesamiento complejo.

---


In [ ]:
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

## 📑 Dataset Artificial: Ensayo Clínico Farmacológico

**Escenario médico imaginario:** Un grupo de investigadores está probando un nuevo medicamento. Tienen registros de 50 pacientes con tres variables clínicas:
- **Edad_Paciente**: en años
- **Dosis_Diaria_mg**: 20, 50 u 80 mg/día
- **Funcion_Renal_Pct**: porcentaje de función renal (100% = normal, <60% = insuficiencia)
- **Tuvo_Efecto_Secundario**: 0=No, 1=Sí (variable a predecir)

**¿Por qué la función renal importa en farmacología?** La mayoría de los medicamentos se eliminan por los riñones. Si la función renal está comprometida, el fármaco se acumula en la sangre y la probabilidad de efectos tóxicos aumenta. Este es un principio básico de farmacocinética.


In [ ]:
data = {
    'Edad_Paciente': [
        58, 79, 73, 58, 38, 58, 81, 74, 39, 84, 51, 62, 48, 48, 43, 61, 41, 74,
        71, 74, 58, 20, 29, 61, 81, 71, 38, 26, 43, 61, 71, 81, 38, 71, 51, 61,
        38, 62, 53, 51, 58, 61, 38, 20, 81, 62, 43, 61, 29, 39
    ],
    'Dosis_Diaria_mg': [
        80, 50, 80, 20, 50, 20, 80, 50, 80, 80, 80, 50, 50, 80, 80, 20, 50, 50,
        50, 50, 20, 80, 50, 50, 80, 80, 50, 50, 20, 80, 20, 20, 20, 50, 80, 80,
        80, 20, 50, 80, 80, 80, 80, 50, 50, 50, 50, 20, 20, 20
    ],
    'Funcion_Renal_Pct': [
        93, 83, 94, 61, 63, 83, 72, 72, 79, 94, 99, 98, 77, 69, 75, 78, 93, 73,
        64, 97, 61, 93, 85, 96, 75, 84, 86, 70, 78, 62, 91, 69, 89, 87, 89, 79,
        98, 95, 69, 92, 83, 91, 92, 63, 66, 60, 69, 83, 66, 91
    ],
    'Tuvo_Efecto_Secundario': [
        0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0,
        1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0,
        1, 0
    ]
}
df_farmaco = pd.DataFrame(data)
df_farmaco

---

## 📊 EDA del Dataset


In [ ]:
df_farmaco.columns

In [ ]:
df_farmaco.info()

In [ ]:
df_farmaco.describe()

In [ ]:
# Scatter plot: Edad vs Dosis, coloreado por efecto secundario
# Buscamos patrones visuales: ¿los pacientes mayores con dosis altas tienen más efectos?

df_farmaco['Resultado'] = df_farmaco['Tuvo_Efecto_Secundario'].map({0: 'No', 1: 'Sí'})

plt.figure(figsize=(10, 7))
sns.scatterplot(data=df_farmaco, x='Edad_Paciente', y='Dosis_Diaria_mg',
                hue='Resultado', style='Resultado', s=100)
plt.title('Relación entre Edad, Dosis y Efectos Secundarios')
plt.xlabel('Edad del Paciente')
plt.ylabel('Dosis Diaria (mg)')
plt.grid(True)
plt.show()

In [ ]:
# Scatter plot: Función Renal vs Dosis
# Hipótesis: pacientes con menor función renal + dosis alta → más efectos secundarios

plt.figure(figsize=(10, 7))
sns.scatterplot(data=df_farmaco, x='Funcion_Renal_Pct', y='Dosis_Diaria_mg',
                hue='Resultado', style='Resultado', s=100)
plt.title('Función Renal vs Dosis Diaria y Efectos Secundarios')
plt.xlabel('Función Renal (%)')
plt.ylabel('Dosis Diaria (mg)')
plt.grid(True)
plt.show()

---

## ⚙️ Preparación de Datos


In [ ]:
# Separamos features y target
X = df_farmaco[['Edad_Paciente', 'Dosis_Diaria_mg', 'Funcion_Renal_Pct']]
y = df_farmaco['Tuvo_Efecto_Secundario']

# División train (70%) / test (30%) — dataset pequeño, no usamos validación separada
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f"Train: {len(X_train)} muestras | Test: {len(X_test)} muestras")

In [ ]:
print("Tamaño del conjunto de prueba:", y_test.size)

---

## 🔵 SVM — Máquinas de Soporte Vectorial

**Idea intuitiva:** SVM busca el hiperplano (línea en 2D, plano en 3D) que separa las dos clases con el *mayor margen posible*. Los puntos más cercanos a ese hiperplano se llaman **vectores de soporte** y son los únicos que determinan la frontera.

**Kernel:** Permite transformar los datos a un espacio de mayor dimensión donde son separables linealmente. Es la magia que hace que SVM funcione con datos no lineales.

| Kernel | Cuándo usar |
|---|---|
| `linear` | Datos linealmente separables |
| `rbf` (Gaussiano) | Frontera de decisión curva |
| `poly` | Relaciones polinomiales |

**`C`** (parámetro de regularización): controla el equilibrio entre maximizar el margen y clasificar correctamente los puntos de entrenamiento.
- C pequeño → margen grande, más errores en train (underfitting)
- C grande → margen pequeño, pocos errores en train (posible overfitting)

> **Contexto médico 🩺:** SVM se usó exitosamente para clasificar imágenes de resonancia magnética y distinguir tejido sano de canceroso antes de la era del deep learning.


In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler

# IMPORTANTE: SVM es sensible a la escala → siempre escalar antes
print("Escalando los datos...")
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit+transform en train
X_test_scaled  = scaler.transform(X_test)        # solo transform en test (¡nunca fit!)
print("Datos escalados.")

In [ ]:
# Entrenamos SVM con kernel lineal y C=0.3
modelo_svm = SVC(kernel='linear', C=0.3)
print("Entrenando el modelo SVM...")
modelo_svm.fit(X_train_scaled, y_train)
print("¡Entrenamiento completado!")

In [ ]:
# VISUALIZACIÓN DE LA FRONTERA EN 2D USANDO PCA COMO "LENTE"
# Como tenemos 3 características, usamos PCA para aplanarlas a 2D y poder graficar
import numpy as np
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
pca.fit(X_train_scaled)

X_scaled = scaler.transform(X)
X_pca    = pca.transform(X_scaled)

x_min, x_max = X_pca[:, 0].min() - 0.5, X_pca[:, 0].max() + 0.5
y_min, y_max = X_pca[:, 1].min() - 0.5, X_pca[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                     np.arange(y_min, y_max, 0.02))

# Proyectamos la malla 2D al espacio 3D original para consultar al modelo SVM
mesh_points_pca    = np.c_[xx.ravel(), yy.ravel()]
mesh_points_3d     = pca.inverse_transform(mesh_points_pca)

# decision_function devuelve la distancia signada al hiperplano
Z = modelo_svm.decision_function(mesh_points_3d).reshape(xx.shape)

plt.figure(figsize=(12, 9))
# Dibuja el hiperplano (contorno 0) y los márgenes (contornos -1 y +1)
plt.contour(xx, yy, Z, colors=['gray', 'black', 'gray'],
            levels=[-1, 0, 1], linestyles=['--', '-', '--'], linewidths=2)

df_pca = pd.DataFrame(data=X_pca, columns=['PC1', 'PC2'])
df_pca['Resultado'] = df_farmaco['Resultado'].values
sns.scatterplot(data=df_pca, x='PC1', y='PC2',
                hue='Resultado', style='Resultado', s=100, alpha=0.9)

# Marcamos los vectores de soporte (los puntos "difíciles" que definen la frontera)
sv_pca = pca.transform(modelo_svm.support_vectors_)
plt.scatter(sv_pca[:, 0], sv_pca[:, 1], s=250, facecolors='none',
            edgecolors='black', linewidth=1.5, label='Vectores de Soporte')

plt.title('SVM: Frontera de Decisión (Vista con PCA)')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

# Evaluamos el SVM en el conjunto de prueba
predicciones_svm = modelo_svm.predict(X_test_scaled)
precision_svm    = accuracy_score(y_test, predicciones_svm)
print(f"Accuracy del SVM en datos de prueba: {precision_svm * 100:.2f}%")

In [ ]:
# Matriz de confusión del SVM
cm_svm = confusion_matrix(y_test, predicciones_svm)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='viridis',
            xticklabels=['Predicción No', 'Predicción Sí'],
            yticklabels=['Real No', 'Real Sí'])
plt.ylabel('Etiqueta Real')
plt.xlabel('Etiqueta Predicha')
plt.title('Matriz de Confusión del SVM')
plt.show()

---

## 🌳 Árbol de Decisión (Decision Tree)

**Idea intuitiva:** El modelo aprende una serie de preguntas tipo "¿la dosis es mayor a 65 mg?" y construye un árbol binario de decisiones. Cada nodo es una pregunta, cada hoja es una predicción.

**Criterio de división (`criterion`):**
- `gini`: mide la "impureza" de un nodo — qué tan mezcladas están las clases.
- `entropy`: mide la incertidumbre de información (relacionado con la entropía de Shannon).

**`max_depth`:** Limita la profundidad del árbol para evitar overfitting. Un árbol muy profundo memoriza el conjunto de entrenamiento.

> **Ventaja clínica enorme:** Un árbol de decisión es la única arquitectura de ML que un médico puede leer directamente. "Si el paciente tiene más de 60 años Y toma más de 50mg → tiene 70% de probabilidad de efectos secundarios." Esto es clave en medicina, donde los modelos de caja negra generan desconfianza.


In [ ]:
# Árbol de decisión con profundidad máxima 2 (para que sea visualizable)
# criterion='entropy' → usa información como criterio de separación
arbol_farmaco = DecisionTreeClassifier(criterion='entropy', max_depth=2)
print("Entrenando el Árbol de Decisión...")
arbol_farmaco.fit(X_train, y_train)
print("¡Entrenamiento completado!")

In [ ]:
predicciones_arbol = arbol_farmaco.predict(X_test)
precision_arbol    = accuracy_score(y_test, predicciones_arbol)
print(f"Accuracy del Árbol de Decisión: {precision_arbol * 100:.2f}%")

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

cm = confusion_matrix(y_test, predicciones_arbol)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicción No', 'Predicción Sí'],
            yticklabels=['Real No', 'Real Sí'])
plt.ylabel('Etiqueta Real')
plt.xlabel('Etiqueta Predicha')
plt.title('Matriz de Confusión — Árbol de Decisión')
plt.show()

In [ ]:
# VISUALIZACIÓN DEL ÁRBOL
# Esta es la gran ventaja del árbol sobre otros modelos: ¡puedes leerlo!
# Cada nodo muestra: la pregunta, la impureza (entropy), las muestras y la distribución de clases

print("Visualizando el árbol de factores de riesgo...")
plt.figure(figsize=(20, 15))
plot_tree(arbol_farmaco,
          feature_names=X.columns,
          class_names=['Sin Efectos', 'Con Efectos'],
          filled=True,     # colorea los nodos según la clase dominante
          rounded=True,
          fontsize=10)
plt.title("Árbol de Decisión: Factores de Riesgo para Efectos Secundarios")
plt.show()

---

## 🌲 Random Forest (Bosque Aleatorio)

**Idea:** Un solo árbol de decisión puede ser inestable (cambia mucho con pequeñas variaciones en los datos). Random Forest construye *muchos* árboles con muestras y características aleatorias, y combina sus predicciones por votación mayoritaria.

**Técnica de bootstrap:** Cada árbol se entrena con una muestra *con reemplazo* del dataset original (~63% de los datos). El 37% restante ("out-of-bag") se usa para estimar el error sin necesidad de un conjunto de validación separado.

**Importancia de características:** Random Forest calcula automáticamente qué tan útil fue cada variable para las predicciones — promediando la reducción de impureza en todos los árboles donde se usó.

> **Dato curioso 🌲:** El Random Forest fue propuesto por Leo Breiman y Adele Cutler en 2001. El nombre "Random Forest" está registrado como marca por ellos. En el campo médico, es uno de los algoritmos más usados por su robustez y su capacidad de manejar datos ruidosos sin mucho preprocesamiento.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# n_estimators=100: construimos 100 árboles
# criterion='gini': criterio de impureza Gini
modelo_rf = RandomForestClassifier(n_estimators=100, criterion='gini')
print("Entrenando el Random Forest...")
modelo_rf.fit(X_train, y_train)
print("¡Entrenamiento completado!")

In [ ]:
predicciones_rf = modelo_rf.predict(X_test)
precision_rf    = accuracy_score(y_test, predicciones_rf)
print(f"Accuracy del Random Forest: {precision_rf * 100:.2f}%")

In [ ]:
cm_rf = confusion_matrix(y_test, predicciones_rf)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicción No', 'Predicción Sí'],
            yticklabels=['Real No', 'Real Sí'])
plt.ylabel('Etiqueta Real')
plt.xlabel('Etiqueta Predicha')
plt.title('Matriz de Confusión del Random Forest')
plt.show()

In [ ]:
# IMPORTANCIA DE CARACTERÍSTICAS
# ¿Qué variable fue más útil para el bosque?
# Valores más altos = más útil para separar las clases

importancias          = modelo_rf.feature_importances_
nombres_caracteris    = X.columns
df_importancias       = pd.DataFrame({'Característica': nombres_caracteris,
                                       'Importancia': importancias})
df_importancias       = df_importancias.sort_values(by='Importancia', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importancia', y='Característica', data=df_importancias, palette='viridis')
plt.title('Importancia de cada Característica en el Random Forest')
plt.xlabel('Nivel de Importancia (reducción de impureza Gini)')
plt.ylabel('Característica')
plt.show()

---

## 🔵 KNN — K Vecinos Más Cercanos (K-Nearest Neighbors)

**Idea:** Para clasificar un punto nuevo, KNN mira los K puntos de entrenamiento más cercanos y asigna la clase mayoritaria entre ellos.

**¿Cómo mide "cercanía"?** Por defecto, usa la distancia Euclidiana en el espacio de características. Por eso es **obligatorio escalar los datos**: si una variable va de 0 a 2000 y otra de 0 a 1, la primera domina la distancia.

**K:** Si K es muy pequeño → modelo muy sensible al ruido (overfitting). Si K es muy grande → pierde detalle local (underfitting).

**Regla práctica:** `K = √(n_muestras)` es un buen punto de partida.

> **Dato histórico:** KNN fue uno de los primeros algoritmos de ML, propuesto en 1951. No "aprende" nada en el sentido tradicional: solo memoriza los datos de entrenamiento y calcula distancias al hacer predicciones. Por eso se llama algoritmo *lazy* (perezoso).


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler

# Escalamos de nuevo (MinMaxScaler) para KNN
print("Escalando los datos para KNN...")
scaler     = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
print("Datos escalados.")

In [ ]:
# Veamos cómo lucen los datos escalados
pd.DataFrame(X_train_scaled, columns=X.columns)

In [ ]:
# n_neighbors=3: considera los 3 vecinos más cercanos para votar
# ¡Experimenta cambiando este número! ¿Qué pasa con K=1? ¿K=10?
modelo_knn = KNeighborsClassifier(n_neighbors=3)
print("Entrenando el modelo KNN...")
modelo_knn.fit(X_train_scaled, y_train)
print("¡Entrenamiento completado!")

In [ ]:
predicciones_knn = modelo_knn.predict(X_test_scaled)
precision_knn    = accuracy_score(y_test, predicciones_knn)
print(f"Accuracy del KNN en datos de prueba: {precision_knn * 100:.2f}%")

cm_knn = confusion_matrix(y_test, predicciones_knn)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicción No', 'Predicción Sí'],
            yticklabels=['Real No', 'Real Sí'])
plt.ylabel('Etiqueta Real')
plt.xlabel('Etiqueta Predicha')
plt.title('Matriz de Confusión del KNN')
plt.show()

---

## 🧮 Gaussian Naive Bayes

**Teorema de Bayes:** `P(clase | features) ∝ P(features | clase) × P(clase)`

El modelo estima `P(features | clase)` asumiendo que cada característica sigue una distribución Gaussiana (normal) y que son **independientes entre sí** (la parte "naive").

**¿Es realista esa independencia?** No siempre. En medicina, la edad y la función renal están correlacionadas. Sin embargo, a pesar de esta suposición incorrecta, Naive Bayes frecuentemente da buenos resultados en la práctica.

**Ventajas médicas:**
- Funciona bien con pocos datos (ideal para datasets médicos pequeños)
- Muy rápido de entrenar
- No requiere escalado
- Probabilidades calibradas (útil para tomar decisiones con umbral de riesgo)


In [ ]:
# Primero: visualizamos la distribución de cada variable por clase
# Si las distribuciones son diferentes entre clases → la variable es informativa

sns.set(style="whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Distribución de Características por Resultado', fontsize=16)

sns.histplot(data=df_farmaco, x='Edad_Paciente',     hue='Resultado', kde=True, ax=axes[0])
axes[0].set_title('Distribución de la Edad')

sns.histplot(data=df_farmaco, x='Dosis_Diaria_mg',   hue='Resultado', kde=True, ax=axes[1])
axes[1].set_title('Distribución de la Dosis Diaria')

sns.histplot(data=df_farmaco, x='Funcion_Renal_Pct', hue='Resultado', kde=True, ax=axes[2])
axes[2].set_title('Distribución de la Función Renal')

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.naive_bayes import GaussianNB

# Gaussian NB: no requiere escalado porque modela la distribución internamente
modelo_gnb = GaussianNB()
print("Entrenando Gaussian Naive Bayes...")
modelo_gnb.fit(X_train, y_train)
print("¡Entrenamiento completado!")

In [ ]:
predicciones_gnb = modelo_gnb.predict(X_test)
precision_gnb    = accuracy_score(y_test, predicciones_gnb)
print(f"Accuracy del Gaussian NB: {precision_gnb * 100:.2f}%")

In [ ]:
cm_gnb = confusion_matrix(y_test, predicciones_gnb)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_gnb, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Predicción No', 'Predicción Sí'],
            yticklabels=['Real No', 'Real Sí'])
plt.ylabel('Etiqueta Real')
plt.xlabel('Etiqueta Predicha')
plt.title('Matriz de Confusión del Gaussian Naive Bayes')
plt.show()

---

## 🧠 Red Neuronal Artificial (MLP con sklearn)

**Perceptrón Multicapa:** Varias capas de neuronas artificiales conectadas. Cada neurona calcula una combinación lineal de sus entradas y pasa el resultado por una **función de activación** no lineal.

```
Entrada → [Capa Oculta 1: 10 neuronas] → [Capa Oculta 2: 5 neuronas] → Salida
```

**ReLU** (Rectified Linear Unit): la función de activación más popular actualmente. Simple pero poderosa: `f(x) = max(0, x)`.

> **¿Por qué se necesitan funciones de activación?** Sin ellas, una red neuronal de cualquier profundidad sería equivalente a una sola capa lineal. Las funciones no lineales permiten al modelo aprender relaciones complejas.

> **Contexto médico:** Las redes neuronales profundas (deep learning) han revolucionado el diagnóstico por imagen médica. En dermatología, ciertos modelos igualan o superan la precisión de dermatólogos expertos para detectar melanoma en fotos de piel.


In [ ]:
from sklearn.neural_network import MLPClassifier

# hidden_layer_sizes=(10, 5): dos capas ocultas con 10 y 5 neuronas respectivamente
# max_iter=1000: suficientes iteraciones para que converja
modelo_red_neuronal = MLPClassifier(
    hidden_layer_sizes=(10, 5),
    max_iter=1000,
    random_state=42
)

# Las redes neuronales REQUIEREN escalado previo
modelo_red_neuronal.fit(X_train_scaled, y_train)
print("¡Entrenamiento de la Red Neuronal completado!")

In [ ]:
predicciones_nn = modelo_red_neuronal.predict(X_test_scaled)
precision_nn    = accuracy_score(y_test, predicciones_nn)
print(f"Accuracy de la Red Neuronal: {precision_nn * 100:.2f}%")

In [ ]:
cm_nn = confusion_matrix(y_test, predicciones_nn)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_nn, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Predicción No', 'Predicción Sí'],
            yticklabels=['Real No', 'Real Sí'])
plt.ylabel('Etiqueta Real')
plt.xlabel('Etiqueta Predicha')
plt.title('Matriz de Confusión de la Red Neuronal (MLP)')
plt.show()

---

## 📊 Comparativa Final de Modelos

| Modelo | Fortalezas en medicina | Debilidades |
|---|---|---|
| SVM | Funciona bien con pocos datos | Difícil de interpretar, lento con datasets grandes |
| Árbol de Decisión | Totalmente interpretable | Inestable, overfitting fácil |
| Random Forest | Robusto, da importancia de variables | Caja negra parcial |
| KNN | Sin suposiciones distribucionales | Lento en predicción, sensible a escala |
| Naive Bayes | Funciona con pocos datos, probabilidades | Asume independencia (rara en medicina) |
| Red Neuronal | Captura relaciones complejas | Necesita muchos datos, caja negra total |

> **Lección clave 🎓:** No existe el "mejor modelo" universal. La elección depende del tamaño del dataset, la interpretabilidad requerida, el tiempo disponible y las consecuencias de los errores (un FN en oncología puede ser fatal).
